In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM


df = pd.read_csv(
    r"C:\A-CMSI research\Data\HMM data\HMM_Input_features.csv"
)

# =====================================
# Features
# =====================================

features = [
    "SPY_Return",
    "SPY_V",
    "RollingVol21",
    "RollingSkew21",
    "Drawdown",
    "VolOfVol",
    "VIX",
    "C_t"
]

X = df[features]


scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


best_model = None
best_score = -np.inf

for seed in range(30):

    model = GaussianHMM(
        n_components=5,
        covariance_type="full",
        n_iter=200,
        random_state=seed
    )

    model.fit(X_scaled)

    score = model.score(X_scaled)

    if score > best_score:

        best_score = score
        best_model = model

print("="*60)
print("Best Log-Likelihood:", best_score)
print("="*60)


hidden_states = best_model.predict(X_scaled)

df["State"] = hidden_states


print("\nSTATE MEANS\n")

print(
    df.groupby("State")[[
        "SPY_Return",
        "SPY_V",
        "RollingVol21",
        "RollingSkew21",
        "Drawdown",
        "VolOfVol",
        "VIX",
        "C_t"
    ]].mean()
)


print("\nTRANSITION MATRIX\n")

print(best_model.transmat_)


logL = best_model.score(X_scaled)

n_states = best_model.n_components
n_features = X_scaled.shape[1]
n_samples = X_scaled.shape[0]


k = (
    (n_states - 1)
    + n_states * (n_states - 1)
    + n_states * n_features
    + n_states * n_features * (n_features + 1) / 2
)

AIC = -2 * logL + 2 * k

BIC = -2 * logL + np.log(n_samples) * k

print("\nMODEL SELECTION")

print("----------------------------")
print("Number of States :", n_states)
print("Log-Likelihood   :", logL)
print("Parameters       :", int(k))
print("AIC              :", AIC)
print("BIC              :", BIC)
print("----------------------------")

Best Log-Likelihood: -12642.887125158517

STATE MEANS

       SPY_Return     SPY_V  RollingVol21  RollingSkew21  Drawdown  VolOfVol  \
State                                                                          
0        0.000584 -0.138751      0.011233       0.093896 -0.107400  0.001111   
1        0.001167 -0.713761      0.005392      -0.012573 -0.003389  0.000622   
2       -0.000672  1.103135      0.010815      -0.601550 -0.041621  0.001782   
3        0.000662  0.606165      0.021074      -0.292386 -0.099498  0.005420   
4        0.000885 -0.631951      0.006657      -0.169163 -0.008732  0.001147   

             VIX       C_t  
State                       
0      19.787400  0.738417  
1      12.567603  0.531674  
2      19.799922  0.745515  
3      31.844286  0.814392  
4      17.688137  0.559814  

TRANSITION MATRIX

[[9.91031550e-001 8.98533258e-053 2.94090745e-023 1.99243028e-003
  6.97601988e-003]
 [3.52090930e-088 9.69368347e-001 9.47246204e-003 8.09207377e-133
  2.115919

In [3]:
print(df["State"].value_counts().sort_index())

State
0    573
1    630
2    515
3    210
4    585
Name: count, dtype: int64
